In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shared.utils as su

In [3]:
data_dir = "/scratch/shared/beegfs/piyush/datasets/SimCSE-NLI"
csv_path = f"{data_dir}/covr/chiral10k-covr10k.csv"

df = pd.read_csv(csv_path)
df.shape

(20000, 4)

In [5]:
df.source.value_counts()

source
covr     10000
nli       9000
ego4d     1000
Name: count, dtype: int64

In [8]:
df_chiral = df[df.source == 'ego4d']
df_chiral.shape

(1000, 4)

In [12]:
# Pick a subset to annotate via an LLM-judge
df_chiral_subset = df_chiral.sample(n=100, random_state=42)

In [13]:
df_chiral_subset.iloc[0].to_dict()

{'sent0': 'The seamstress unfolds the cloth piece with both hands.',
 'sent1': 'The seamstress unfolds the cardboard piece with both hands.',
 'hard_neg': 'The seamstress folds the cloth piece with both hands.',
 'source': 'ego4d'}

### LLM Judgement

In [16]:
from utils.gemini_utils import GeminiWrapper

model_key = "gemini-3-flash-preview"
vlm = GeminiWrapper(model_key=model_key, fps=1.)
vlm.forward_text_only("Add 23 + 45")

Loading gemini-3-flash-preview with FPS=1.0.....................................  


'23 + 45 = **68**'

In [18]:
import json
import re
from tqdm import tqdm

def run_llm_judge(df, prompt_path="chiral_hard_negative_prompt.txt", debug=False):
    with open(prompt_path) as f:
        prompt_template = f.read()

    def fill_prompt(template, row):
        for key in ["sent0", "sent1", "hard_neg"]:
            template = template.replace(f"{{{key}}}", row[key])
        return template

    sample = df.sample(1) if debug else df
    results = []

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Judging hard negatives"):
        prompt = fill_prompt(prompt_template, row)

        if debug:
            print("=" * 60)
            print(f"  sent0:    {row['sent0']}")
            print(f"  sent1:    {row['sent1']}")
            print(f"  hard_neg: {row['hard_neg']}")
            print(f"\n--- PROMPT ---\n{prompt}\n")

        try:
            raw = vlm.forward_text_only(prompt)
            clean = re.sub(r"```json|```", "", raw).strip()
            result = json.loads(clean)
        except Exception as e:
            result = {"error": str(e), "raw": raw}

        if debug:
            print("--- OUTPUT ---")
            print(json.dumps(result, indent=2))
            print("=" * 60)

        results.append({**row.to_dict(), **result})

    return pd.DataFrame(results)


run_llm_judge(df_chiral_subset, debug=True)

Judging hard negatives:   0%|                                                                                                          | 0/1 [00:00<?, ?it/s]

  sent0:    The hiker puts the water bottle on the kitchen surface
  sent1:    The hiker puts the plastic bottle in the cabinet
  hard_neg: The hiker takes the water bottle off the kitchen surface

--- PROMPT ---
You are evaluating the quality of a hard-negative sentence pair for a semantic similarity dataset.

Given:
  Anchor:    "The hiker puts the water bottle on the kitchen surface"
  Positive:  "The hiker puts the plastic bottle in the cabinet"
  Hard-neg:  "The hiker takes the water bottle off the kitchen surface"

A *good* hard negative must satisfy BOTH conditions:

[A] STATIC PRESERVATION — The non-action elements (subject, object, setting, manner)  
    are identical or near-identical between the anchor and the hard negative.

[B] CHIRAL ACTION — The action in the hard negative is the temporal/directional antonym  
    of the anchor's action (e.g., fold↔unfold, open↔close, pick up↔put down).  
    It must NOT be a different action, a negation ("not folding"), or a paraphrase.

Judging hard negatives: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.56s/it]

--- OUTPUT ---
{
  "static_score": 3,
  "static_reason": "The subject, object, and location are identical across the anchor and the hard negative.",
  "chiral_score": 3,
  "chiral_reason": "'Takes off' is the direct directional and temporal antonym of 'puts on'.",
  "verdict": "PASS"
}


,sent0,sent1,hard_neg,source,static_score,static_reason,chiral_score,chiral_reason,verdict
0,The hiker puts the water bottle on the kitchen...,The hiker puts the plastic bottle in the cabinet,The hiker takes the water bottle off the kitch...,ego4d,3,"The subject, object, and location are identica...",3,'Takes off' is the direct directional and temp...,PASS


In [43]:
df_chiral_subset_gemini_results = run_llm_judge(df_chiral_subset.sample(n=20))
df_chiral_subset_gemini_results.shape

Judging hard negatives: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:22<00:00,  7.14s/it]


(20, 9)

In [49]:
df_chiral_subset_gemini_results.sample(n=1).iloc[0].to_dict()

{'sent0': 'The baker picks bread',
 'sent1': 'The woman H picks a bread from the bowl on the table with her right hand.',
 'hard_neg': 'The baker puts bread down',
 'source': 'ego4d',
 'static_score': 3,
 'static_reason': 'The subject and object are identical across both sentences with no additional modifiers or setting changes.',
 'chiral_score': 3,
 'chiral_reason': "The action 'puts down' is a perfect directional antonym to 'picks' in the context of handling an object.",
 'verdict': 'PASS'}

In [21]:
df_chiral_subset_gemini_results.verdict.value_counts()

verdict
PASS    100
Name: count, dtype: int64

In [24]:
df_chiral_subset_gemini_results.static_score.unique(), df_chiral_subset_gemini_results.chiral_score.unique()

(array([3]), array([3]))

In [42]:
# Quick diagnostic — run on 5 samples and print raw responses
prompt_path = "chiral_hard_negative_prompt.txt"

with open(prompt_path) as f:
    prompt_template = f.read()

def fill_prompt(template, row):
    for key in ["sent0", "sent1", "hard_neg"]:
        template = template.replace(f"{{{key}}}", row[key])
    return template

# for _, row in df_chiral_subset.sample(5).iterrows():
row = df_chiral_subset.iloc[4]
print(row)
raw = vlm.forward_text_only(fill_prompt(prompt_template, row))
print(raw, "\n---")

sent0        The woman opens the box with her left hand.
sent1             The gardener opens the garden tool box
hard_neg    The woman closes the box with her left hand.
source                                             ego4d
Name: 9411, dtype: object
{
  "static_score": 3,
  "static_reason": "All non-action elements, including the subject, object, and specific manner, are identical between the anchor and the hard negative.",
  "chiral_score": 3,
  "chiral_reason": "The actions 'opens' and 'closes' are perfect temporal antonyms representing the reversal of the same movement.",
  "verdict": "PASS"
} 
---


sent0        The woman opens the box with her left hand.
sent1             The gardener opens the garden tool box
hard_neg    The woman closes the box with her left hand.
source                                             ego4d
Name: 9411, dtype: object

In [30]:
df_chiral_subset_gemini_results

,sent0,sent1,hard_neg,source,static_score,static_reason,chiral_score,chiral_reason,verdict
0,The seamstress unfolds the cloth piece with bo...,The seamstress unfolds the cardboard piece wit...,The seamstress folds the cloth piece with both...,ego4d,3,"The subject, object, and manner are identical ...",3,The action 'folds' is the direct directional a...,PASS
1,The **waiter** picks up the spoon,The **waiter** Picks up a spoon from a kitchen...,The **waiter** puts down the spoon,ego4d,3,The subject 'The waiter' and the object 'the s...,3,The action 'puts down' is the direct temporal ...,PASS
2,The mechanic puts his left hand in the motorcycle,The mechanic puts her hand on the book,The mechanic takes his left hand out of the mo...,ego4d,3,"The subject, object, and manner ('his left han...",3,The action 'takes out' is the direct direction...,PASS
3,The barista drops the glass cup on the kitchen...,The barista drops the first cup lid into the sink,The barista picks up the glass cup from the ki...,ego4d,3,"The subject, object, and setting are identical...",3,Picking up an object from a surface is the dir...,PASS
4,The woman opens the box with her left hand.,The gardener opens the garden tool box,The woman closes the box with her left hand.,ego4d,3,"The subject, object, and manner in the hard ne...",3,The action 'closes' is the direct temporal ant...,PASS
...,...,...,...,...,...,...,...,...,...
95,The carpenter puts the electric hand saw into ...,The carpenter puts hand on a stand,The carpenter takes the electric hand saw out ...,ego4d,3,"The subject, object, and setting are identical...",3,'Takes out of' is the direct directional and t...,PASS
96,The student picks another book from the bookshelf,The boy picks the book from the armrest of the...,The student puts another book back on the book...,ego4d,3,"The subject, object, and setting remain identi...",3,'Picks from' and 'puts back on' are direct tem...,PASS
97,The seamstress picks the scissors from the cab...,The seamstress picks up the scissors,The seamstress puts the scissors into the cabi...,ego4d,3,"The subject, object, and setting are identical...",3,The actions 'picks from' and 'puts into' are d...,PASS
98,The woman closes the bottle,The woman closes a bottle of drink,The woman opens the bottle,ego4d,3,The subject and object are identical between t...,3,The action 'opens' is the direct temporal and ...,PASS


In [51]:
from pigeon import annotate
from IPython.display import display, HTML

def run_human_annotation(df):
    examples = list(df.itertuples())  # keeps index so we can merge back

    def fmt(entry):
        display(HTML(f"""
        <div style="font-family: monospace; font-size: 14px; line-height: 2">
            <b>anchor:</b>   {entry.sent0}<br>
            <b>hard_neg:</b> <span style="color: #c0392b">{entry.hard_neg}</span>
        </div>"""))

    def annotate_axis(question, options):
        print(f"\n{'='*60}\n📝 {question}\n{'='*60}")
        return annotate(examples, options=options, display_fn=fmt)

    static_labels = annotate_axis(
        "[STATIC] Are the subject, object, and setting preserved between anchor and hard_neg?",
        options=["3 - fully preserved", "2 - minor drift", "1 - violated"]
    )
    chiral_labels = annotate_axis(
        "[CHIRAL] Is the action a true temporal/directional antonym (e.g. fold↔unfold)?",
        options=["3 - correct antonym", "2 - partial", "1 - wrong action"]
    )

    # pigeon returns [(example, label), ...] — merge back via the original index
    static_map  = {row.Index: label for row, label in static_labels}
    chiral_map  = {row.Index: label for row, label in chiral_labels}

    df = df.copy()
    df["human_static"]  = df.index.map(static_map)   # NaN if not annotated
    df["human_chiral"]  = df.index.map(chiral_map)

    annotated = df.dropna(subset=["human_static", "human_chiral"]).copy()
    
    score = lambda col: col.map(lambda x: int(x[0]) if isinstance(x, str) else 0)
    
    annotated["human_verdict"] = (
        (score(annotated["human_static"]) >= 2) &
        (score(annotated["human_chiral"]) >= 2)
    ).map({True: "PASS", False: "FAIL"})

    print(f"\n✅ Annotated {len(annotated)}/{len(df)} examples")
    return annotated


df_annotated = run_human_annotation(df_chiral_subset.sample(n=50))


📝 [STATIC] Are the subject, object, and setting preserved between anchor and hard_neg?


HTML(value='0 examples annotated, 51 examples left')

Output()


📝 [CHIRAL] Is the action a true temporal/directional antonym (e.g. fold↔unfold)?


HTML(value='0 examples annotated, 51 examples left')

Output()


✅ Annotated 0/50 examples
